# Tome 12, Chapter 3: The geometry of numbers (Python companion)

Exercises first, solutions at the end. We compute Minkowski bounds and class groups, find two-square decompositions as short lattice vectors, look at the class numbers of imaginary quadratic fields, and implement the LLL lattice reduction algorithm, the workhorse of computational number theory and of the attacks (and constructions) of lattice cryptography. Requires `cypari2` and `numpy`.

In [ ]:
import math, random, time
from fractions import Fraction
import cypari2
pari = cypari2.Pari()
pari.allocatemem(4 * 10**8, silent=True)

## Exercises

**P3.1 (★) Minkowski bounds and class groups.** For $\mathbb Q(\sqrt{-5})$, $\mathbb Q(\sqrt{-23})$, $\mathbb Q(\sqrt{-47})$, $\mathbb Q(\sqrt{10})$, $\mathbb Q(\sqrt[3]2)$, $\mathbb Q(\sqrt[3]{10})$ and $\mathbb Q(\sqrt[3]7)$, compute $M_K$ (Theorem 3.9), the prime numbers below it, and the class group with PARI's `bnfinit`. Compare with Exercises 3.3–3.7.

**P3.2 (★★) Two squares as a short vector.** For a prime $p\equiv1\pmod4$, the lattice $L=\{(a,b):a\equiv ub\pmod p\}$ of Example 3.4 has basis $(p,0),(u,1)$. Implement Lagrange–Gauss reduction of a $2$-dimensional basis (repeatedly subtract from the longer vector the nearest integer multiple of the shorter) and show that the shortest vector gives $p=a^2+b^2$. Try it on $p=10^{30}+57$ (check that it is prime and $\equiv1\bmod 4$).

**P3.3 (★★) Class numbers of imaginary quadratic fields.** Compute $h(d)$ for all fundamental discriminants $-10^4<d<0$. Find those with $h=1$. Plot or tabulate $h(d)/\sqrt{|d|}$: how large and how small does it get? (Chapter 8 proves $h(d)=\frac{w\sqrt{|d|}}{2\pi}L(1,\chi_d)$.)

**P3.4 (★★★) CS application: LLL.** Implement the Lenstra–Lenstra–Lovász algorithm with exact rational arithmetic. Use it to

1. recover the minimal polynomial of $\alpha=\sqrt2+\sqrt[3]3$ from a $60$-digit approximation, by finding a short vector in the lattice spanned by the rows of $\bigl(I_7\mid C\alpha^k\bigr)_{0\le k\le6}$ with $C=10^{50}$;
2. break a toy *knapsack cryptosystem*: given random weights $w_1,\dots,w_n$ of about $2n$ bits and $s=\sum_{i\in A}w_i$ for a secret set $A$, recover $A$ (Lagarias–Odlyzko).

Why does a short vector of the first lattice give an integer relation $\sum c_k\alpha^k=0$?

## Solutions

### P3.1
**How to find it.** $M_K=(4/\pi)^{r_2}\,n!/n^n\,\sqrt{|d_K|}$; PARI gives $r_2$ and $d_K$ (`nf.sign`, `nf.disc`) and the class group (`bnf.cyc`, the orders of its cyclic factors).

In [ ]:
fields = ['x^2+5', 'x^2-x+6', 'x^2-x+12', 'x^2-10', 'x^3-2', 'x^3-10', 'x^3-7']
for f in fields:
    bnf = pari.bnfinit(f)
    n = int(pari.poldegree(f))
    d = int(pari('nfinit(%s).disc' % f))
    r2 = int(pari('nfinit(%s).sign[2]' % f))
    M = (4 / math.pi)**r2 * math.factorial(n) / n**n * math.sqrt(abs(d))
    small = [int(p) for p in pari.primes(int(M) + 1) if p <= M]
    print("%-9s d_K=%5d  M_K=%5.2f  primes %s  class group %s" % (f, d, M, small, pari('bnfinit(%s).cyc' % f)))

The class groups are $\mathbb Z/2$, $\mathbb Z/3$, $\mathbb Z/5$, $\mathbb Z/2$, trivial, trivial and $\mathbb Z/3$, as in the text and the exercises ($x^2-x+6$ and $x^2-x+12$ define $\mathbb Q(\sqrt{-23})$ and $\mathbb Q(\sqrt{-47})$). In every case but one only the primes $2$ and $3$ need to be examined; for $\mathbb Q(\sqrt[3]7)$, with $d_K=-1323$, the bound is about $10.3$ and the primes $2,3,5,7$ must be looked at.

### P3.2
**How to find it.** Lagrange–Gauss reduction is Euclid's algorithm for $2$-dimensional lattices; it ends with a shortest vector. The lattice has covolume $p$, and the minimum of $a^2+b^2$ on $L\setminus\{0\}$ is $p$ (Example 3.4), so the shortest vector is a decomposition. To find $u$, take $c^{(p-1)/4}$ for a quadratic non-residue $c$.

In [ ]:
def gauss_reduce(v, w):
    norm = lambda x: x[0]**2 + x[1]**2
    if norm(v) > norm(w):
        v, w = w, v
    while True:
        m = round(Fraction(v[0]*w[0] + v[1]*w[1], norm(v)))
        w = (w[0] - m*v[0], w[1] - m*v[1])
        if norm(w) >= norm(v):
            return v, w
        v, w = w, v

def two_squares(p):
    c = 2
    while pow(c, (p - 1) // 2, p) != p - 1:
        c += 1
    u = pow(c, (p - 1) // 4, p)
    v, _ = gauss_reduce((p, 0), (u, 1))
    return abs(v[0]), abs(v[1])

for p in (13, 101, 10**30 + 57):
    assert pari.isprime(p) and p % 4 == 1
    a, b = two_squares(p)
    print(p, "=", a, "^2 +", b, "^2:", a*a + b*b == p)

The shortest vector always gives the decomposition, in a few dozen steps even for a $31$-digit prime: Lagrange–Gauss reduction runs in $O(\log p)$ iterations, like Euclid's algorithm. This is essentially Cornacchia's algorithm, and it is the two-dimensional case of LLL.

### P3.3
**How to find it.** `isfundamental` and `qfbclassno` (or `quadclassunit`) do the work.

In [ ]:
hs = {}
for d in range(-3, -10**4, -1):
    if pari.isfundamental(d):
        hs[d] = int(pari.qfbclassno(d))
print("fundamental discriminants:", len(hs))
print("class number one:", sorted((d for d in hs if hs[d] == 1), reverse=True))
ratios = {d: hs[d] / math.sqrt(-d) for d in hs}
dmax = max(ratios, key=ratios.get); dmin = min(ratios, key=ratios.get)
print("largest h/sqrt|d|: d=%d, h=%d, ratio %.3f" % (dmax, hs[dmax], ratios[dmax]))
print("smallest h/sqrt|d|: d=%d, h=%d, ratio %.3f" % (dmin, hs[dmin], ratios[dmin]))
for lo, hi in ((-1000, -1), (-10**4, -9000)):
    sub = [ratios[d] for d in hs if lo <= d <= hi]
    print("mean of h/sqrt|d| for %d<=d<=%d: %.3f" % (lo, hi, sum(sub) / len(sub)))

Exactly nine fundamental discriminants have $h=1$: $-3,-4,-7,-8,-11,-19,-43,-67,-163$, the *Heegner numbers*; Heegner (1952), Baker and Stark (1967) proved there are no others, solving a problem of Gauss. In this range the ratio $h/\sqrt{|d|}$ goes from about $0.08$ (at $d=-163$) to about $1.46$ (at $d=-8399$, with $h=134$), and averages about $0.46$. By the class number formula (Chapter 8) it is $\frac{w}{2\pi}L(1,\chi_d)$ with $w=2$ for $d<-4$, so $L(1,\chi_d)$ averages about $1.45$; Tome 14 studies its distribution. The extremes come from $d$ for which the small primes are all inert, like $-163$ (small $L(1,\chi_d)=\prod_p(1-\chi_d(p)/p)^{-1}$), or mostly split, like $-8399$ (large $L(1,\chi_d)$).

### P3.4
**How to find it.** LLL keeps a Gram–Schmidt orthogonalisation of the basis, size-reduces each vector against the previous ones, and swaps two neighbours when the second is much shorter than the first in the orthogonal direction (the Lovász condition with $\delta=\frac34$). For the integer relation, a vector $(c_0,\dots,c_6,\ C\sum c_k\alpha^k)$ is short only if $\sum c_k\alpha^k$ is tiny, which for small integers $c_k$ means it is $0$. For the knapsack, the vector $(x_1,\dots,x_n,0)$ with $x_i\in\{0,1\}$ the indicator of $A$ lies in the lattice spanned by the rows $(e_i,\ Nw_i)$ and $(0,\dots,0,\ -Ns)$, and it is unusually short.

In [ ]:
def lll(B, delta=Fraction(3, 4)):
    B = [list(map(Fraction, b)) for b in B]
    n = len(B)
    dot = lambda u, v: sum(x * y for x, y in zip(u, v))
    def gso():
        Bs, mu = [], [[Fraction(0)] * n for _ in range(n)]
        for i in range(n):
            v = B[i][:]
            for j in range(i):
                mu[i][j] = dot(B[i], Bs[j]) / dot(Bs[j], Bs[j])
                v = [a - mu[i][j] * b for a, b in zip(v, Bs[j])]
            Bs.append(v)
        return Bs, mu
    Bs, mu = gso()
    k = 1
    while k < n:
        for j in range(k - 1, -1, -1):
            q = round(mu[k][j])
            if q:
                B[k] = [a - q * b for a, b in zip(B[k], B[j])]
                Bs, mu = gso()
        if dot(Bs[k], Bs[k]) >= (delta - mu[k][k-1]**2) * dot(Bs[k-1], Bs[k-1]):
            k += 1
        else:
            B[k], B[k-1] = B[k-1], B[k]
            Bs, mu = gso()
            k = max(k - 1, 1)
    return [[int(x) for x in b] for b in B]

In [ ]:
from mpmath import mp, mpf, sqrt, cbrt
mp.dps = 60
alpha = sqrt(2) + cbrt(3)
C = 10**50
rows = [[1 if i == j else 0 for j in range(7)] + [int(mp.nint(C * alpha**i))] for i in range(7)]
t = time.perf_counter()
red = lll(rows)
print("LLL in %.2f s" % (time.perf_counter() - t))
coeffs = red[0][:7]
print("relation found:", coeffs)
print("PARI's algdep:", pari('algdep(sqrt(2) + 3^(1/3), 6)'))
x = pari('x')
print("as a polynomial:", sum(c * x**k for k, c in enumerate(coeffs)))

In [ ]:
def knapsack(n, seed):
    rnd = random.Random(seed)
    w = [rnd.getrandbits(2 * n) for _ in range(n)]
    A = [i for i in range(n) if rnd.random() < 0.5]
    s = sum(w[i] for i in A)
    N = 2**n
    rows = [[1 if i == j else 0 for j in range(n)] + [N * w[i]] for i in range(n)]
    rows.append([0] * n + [-N * s])
    t = time.perf_counter()
    for v in lll(rows):
        for sign in (1, -1):
            x = [sign * c for c in v[:n]]
            if v[n] == 0 and set(x) <= {0, 1} and sum(w[i] for i in range(n) if x[i]) == s:
                return sorted(i for i in range(n) if x[i]) == A, time.perf_counter() - t
    return False, time.perf_counter() - t

for n in (8, 12, 16):
    print("n=%2d:" % n, ["%s in %.1fs" % knapsack(n, seed) for seed in range(3)])

LLL finds a relation with small coefficients; up to sign it is the minimal polynomial $X^6-6X^4-6X^3+12X^2-36X+1$ of $\sqrt2+\sqrt[3]3$ (expand $((X-\sqrt2)^3-3)((X+\sqrt2)^3-3)$ to check it). The precision matters: the relation is detected because $C\sum c_k\alpha^k$ is about $C\cdot10^{-60}$, far smaller than the other coordinates, while any non-relation would make the last coordinate of size about $C\cdot10^{-k}$ for small $k$.

The knapsack secret is recovered in every trial. The weights have $2n$ bits, so the *density* $n/\log_2\max w_i\approx\frac12$ is low, and below about $0.94$ the secret vector is the shortest vector of the lattice with high probability (Lagarias–Odlyzko 1985, Coster et al. 1992). This is how the Merkle–Hellman cryptosystem (1978), one of the first public-key schemes, was broken by Shamir (1982) and then by LLL.

The story turned around: today, lattice problems in high dimension (hundreds to a thousand), where LLL's approximation factor $2^{n/2}$ is useless, are the basis of the post-quantum standards ML-KEM (Kyber) and ML-DSA (Dilithium), chosen by NIST in 2024. Their security rests on the belief that finding short vectors in such lattices is hard even for quantum computers, the opposite of what happens in dimension $2$ (P3.2) or $20$ (this exercise). Chapter 5's notebook looks at the cyclotomic rings these schemes use.